In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

import itertools
from tqdm import tqdm

import shap

In [2]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'Lactate', 'Butyrate', 'Acetate', 'AcGum', 'ArGal',
       'Inulin', 'Pectin', 'Starch', 'Xylan'], dtype='<U8')

In [3]:
# data with initial and end point measurements
df_mono = pd.read_csv("data/exp0/exp0_mono_reps.csv")
df_exp0 = pd.read_csv("data/exp0/exp0_comm.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")

# import validation data
df_val = pd.read_csv("data/exp4/exp4_metabolites_best_reps.csv")
df_bad = pd.read_csv("data/exp4/exp4_metabolites_new_worst.csv")
df_bst = pd.read_csv("data/exp4/exp4_metabolites_new_best.csv")

# make metabolite initial condition 0 instead of NaN 
t0_inds = df_mono.Time.values == 0
df_mono.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp0.Time.values == 0
df_exp0.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp1.Time.values == 0
df_exp1.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp2.Time.values == 0
df_exp2.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp3.Time.values == 0
df_exp3.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_val.Time.values == 0
df_val.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_bad.Time.values == 0
df_bad.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_bst.Time.values == 0
df_bst.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

In [4]:
# log that ignores zeros
def zlog(x):
    x[x <= 0] = 1
    return np.log(x)

# shannon diversity
def shannon(y):
    y_rel = y / np.sum(y)
    return np.nansum(-zlog(y_rel)*y_rel)

# define objective 
def objective(y):
    # y is measured exp data [n_time, n_species + n_metabolites]
    
    # endpoint shannon diversity
    diversity = shannon(y[-1, :len(species)])
    
    # variance in species abundances in last two passages
    if np.any(np.isnan(y[-2:, :len(species)])):
        instability = np.nan
    else:
        species_stdv = np.std(y[-2:, :len(species)], 0)
        instability  = np.where(species_stdv>0, species_stdv, 0).mean() 
    
    # endpoint butyrate production 
    butyrate =  y[-1, -2]   
    
    return diversity, instability, butyrate

In [5]:
# bin the last measurement time 
Time = df_mono['Time'].values
for i, t in enumerate(Time):
    if t > 40:
        Time[i] = 1.
df_mono['Time'] = Time

# bin the last measurement time 
Time = df_exp0['Time'].values
for i, t in enumerate(Time):
    if t > 40:
        Time[i] = 1.
df_exp0['Time'] = Time

In [6]:
# concatenate dataframes
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3, df_val, df_bad, df_bst))

In [7]:
# take mean of replicates
df_mean = []

for exp_name, df_exp in df.groupby("Experiments"):
    df_groups = df_exp.groupby("Time")
    df_avg = df_groups[system_variables].mean().reset_index()
    df_avg.insert(0, "Experiments", [exp_name]*df_avg.shape[0])
    df_mean.append(df_avg)
    
df = pd.concat(df_mean)

In [8]:
# determine names of experimental conditions 
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

In [9]:
# add mono culture just to training data 
test_df = df.copy()
train_df = pd.concat((df_mono, df))

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(train_df)
train_df_scaled = scaler.transform(train_df.copy())
test_df_scaled = scaler.transform(test_df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(train_df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
test_data = format_data(test_df, species, metabolites, controls, observed=observed)
test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0, alpha_1=1.,
         evd_tol=1e-3)

Total measurements: 27827, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1407.377, Residuals: 0.00337
Loss: 1317.528, Residuals: 0.00479
Loss: 1267.409, Residuals: -0.00294
Loss: 1254.292, Residuals: -0.00121
Loss: 1144.826, Residuals: 0.00133
Loss: 1115.580, Residuals: -0.00157
Loss: 1059.931, Residuals: -0.00072
Loss: 959.464, Residuals: 0.00195
Loss: 901.888, Residuals: 0.00283
Loss: 860.463, Residuals: 0.00035
Loss: 800.567, Residuals: -0.00012
Loss: 768.055, Residuals: 0.00060
Loss: 760.357, Residuals: 0.00094
Loss: 699.928, Residuals: -0.00024
Loss: 694.737, Residuals: 0.00061
Loss: 648.879, Residuals: 0.00056
Loss: 606.708, Residuals: 0.00030
Loss: 603.728, Residuals: 0.00002
Loss: 577.472, Residuals: 0.00016
Loss: 551.501, Residuals: 0.00018
Loss: 547.901, Residuals: -0.00076
Loss: 541.278, Residuals: -0.00068
Loss: 529.664, Residuals: -0.00058
Loss: 513.145, Residuals: -0.00056
Loss: 512.386, Residuals: -0.00058
Loss: 505.164, Residuals: -0.00062
Loss: 495

In [11]:
# concatenate data points
Xs = []
all_exp_names = []

for (T, X, U, Y, exp_names) in train_data_scaled:
    
    all_exp_names.append(exp_names)
    for xi, ui in zip(X, U):
        
        # append design condition
        Xs.append(np.append(xi, ui[0]))
        
# stack 
X = np.stack(Xs)  
all_exp_names = np.concatenate(all_exp_names)

In [12]:
# set each species as receiver
for i, receiver in enumerate(species):

    # create wrapper for brnn to match SHAP model 
    def model(X):

        # X is matrix of [n_samples, n_inputs] 
        # Decompose X into Species/Metabolites and Fibers
        Xsm = X[:, :len(observed)]
        Xf = X[:, len(observed):]

        # matrix of predictions over time
        Xf = np.stack([np.stack(4*[Xfi]) for Xfi in Xf])
        Y = brnn.forward_batch(brnn.params, Xsm, Xf)

        # return endpoint species prediction
        return Y[:, -1, i]

    # compute the SHAP values for the model
    explainer = shap.Explainer(model, X)
    shap_values = explainer(X)

    # name of shap explanations
    shap_features = []
    for j, affecter in enumerate(system_variables):
        shap_features.append(receiver + "<--" + affecter)

    # init df to save shap values
    df_sensitivity = pd.DataFrame()
    df_sensitivity["Experiments"] = all_exp_names

    # add shap values
    for j, shap_feature in enumerate(shap_features):
        df_sensitivity[shap_feature] = shap_values.values[:, j]

    # add design conditions
    for j, sys_var in enumerate(system_variables):
        if sys_var not in metabolites:
            df_sensitivity[sys_var] = X[:, j]
        
    # save df 
    df_sensitivity.to_csv(f"insights/community_shap/{receiver}_shap.csv", index=False)

PermutationExplainer explainer: 1260it [02:46,  7.32it/s]                       
PermutationExplainer explainer: 1260it [02:05,  9.36it/s]                       
PermutationExplainer explainer: 1260it [02:02,  9.59it/s]                       
PermutationExplainer explainer: 1260it [02:00,  9.70it/s]                       
PermutationExplainer explainer: 1260it [02:04,  9.38it/s]                       
PermutationExplainer explainer: 1260it [02:06,  9.27it/s]                       
PermutationExplainer explainer: 1260it [02:02,  9.52it/s]                       
PermutationExplainer explainer: 1260it [02:02,  9.54it/s]                       
PermutationExplainer explainer: 1260it [02:00,  9.70it/s]                       
PermutationExplainer explainer: 1260it [01:59,  9.72it/s]                       
PermutationExplainer explainer: 1260it [01:59,  9.75it/s]                       
PermutationExplainer explainer: 1260it [01:58,  9.79it/s]                       
PermutationExplainer explain